# NB02: Functional augmentation

Tests whether genus-weighted functional features improve prediction beyond CLR taxonomy alone.

**H2**: Genus-weighted functional features improve beyond taxonomy + geochem (M1 vs B3).
- Success: ΔRMSE(B3 − M1) CI excludes 0 for ≥2 of 4 metals.

**H3**: Genus-weighted features outperform CWM representation (M2 vs M3).
- Success: M2 RMSE < M3 RMSE for ≥3 of 4 metals.

Models (spatial block CV):
- **B3**: CLR + pH + lat/lon (XGBoost) — from NB01
- **B4**: CWM only (XGBoost)
- **M1**: CLR + genus-weighted functional (XGBoost)
- **M2**: CLR + genus-weighted functional + env (XGBoost)
- **M3**: CLR + CWM + env (XGBoost)

**Outputs**
- `data/cv_results_models.csv` — per-fold RMSE for B4, M1, M2, M3
- `data/bootstrap_h2.csv` — bootstrap ΔRMSE (B3 − M1)
- `data/oof_predictions.parquet` — out-of-fold predictions for all models

In [ ]:
import sys
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from pathlib import Path

for _cand in [Path.cwd() / 'scripts', Path.cwd().parent / 'scripts']:
    if _cand.exists():
        sys.path.insert(0, str(_cand))
        break

DATA_DIR = next(p for p in [Path.cwd() / 'data', Path.cwd().parent / 'data'] if p.exists())
FIG_DIR  = next(p for p in [Path.cwd() / 'figures', Path.cwd().parent / 'figures'] if p.exists())

from modelling import TARGETS, run_spatial_block_cv, rmse, get_features, _drop_nan_rows, build_xgboost
from evaluation import plot_rmse_comparison

feature_matrix = pd.read_parquet(DATA_DIR / 'feature_matrix.parquet')
blocks = pd.read_csv(DATA_DIR / 'spatial_blocks.csv', index_col=0)['block']
b3_baselines = pd.read_csv(DATA_DIR / 'cv_results_baselines.csv')
print(f'Feature matrix: {feature_matrix.shape}')

## 1. Spatial CV for B4, M1, M2, M3

In [ ]:
all_results = []
oof_parts = []

# B3 OOF predictions needed for H2 bootstrap (B3 per-fold RMSE already in cv_results_baselines.csv)
print('Getting B3 OOF predictions...')
for target in TARGETS:
    if target not in feature_matrix.columns:
        continue
    _, oof = run_spatial_block_cv(
        feature_matrix, target, 'B3', blocks,
        model_type='xgboost', return_oof=True,
    )
    if oof is not None:
        oof_parts.append(oof.rename(f'B3_{target}'))
print('  Done.')

for model_name in ['B4', 'M1', 'M2', 'M3']:
    print(f'Running {model_name}...')
    for target in TARGETS:
        if target not in feature_matrix.columns:
            continue
        res, oof = run_spatial_block_cv(
            feature_matrix, target, model_name, blocks,
            model_type='xgboost', return_oof=True,
        )
        all_results.append(res)
        if oof is not None:
            oof_parts.append(oof.rename(f'{model_name}_{target}'))
    print(f'  Done.')

cv_models = pd.concat(all_results, ignore_index=True)
cv_models.to_csv(DATA_DIR / 'cv_results_models.csv', index=False)

oof_df = pd.concat(oof_parts, axis=1)
oof_df.to_parquet(DATA_DIR / 'oof_predictions.parquet')

# Combine with baselines for full comparison
cv_all = pd.concat([b3_baselines, cv_models], ignore_index=True)
pivot = cv_all.groupby(['model', 'target'])['rmse'].mean().unstack('target').round(4)
print('\nMean spatial-CV RMSE:')
print(pivot)


## 2. H2: Bootstrap ΔRMSE (B3 − M1)

Positive ΔRMSE means M1 is better than B3 (genus-weighted adds beyond CLR+geochem).

In [ ]:
N_BOOT = 1000
rng = np.random.default_rng(42)

h2_records = []
for target in TARGETS:
    if target not in feature_matrix.columns:
        continue
    b3_col = f'B3_{target}'
    m1_col = f'M1_{target}'
    if b3_col not in oof_df.columns or m1_col not in oof_df.columns:
        print(f'  Skipping {target}: OOF columns missing (re-run with B3 OOF if needed)')
        continue

    y = feature_matrix[target]
    valid = y.notna() & oof_df[b3_col].notna() & oof_df[m1_col].notna()
    y_v   = y[valid].values
    pb3_v = oof_df.loc[valid, b3_col].values
    pm1_v = oof_df.loc[valid, m1_col].values

    obs_delta = rmse(y_v, pb3_v) - rmse(y_v, pm1_v)
    boot_deltas = []
    for _ in range(N_BOOT):
        idx = rng.integers(0, len(y_v), len(y_v))
        boot_deltas.append(rmse(y_v[idx], pb3_v[idx]) - rmse(y_v[idx], pm1_v[idx]))
    lo, hi = np.percentile(boot_deltas, [2.5, 97.5])
    h2_records.append({
        'target': target,
        'delta_rmse_b3_minus_m1': obs_delta,
        'ci_lo': lo, 'ci_hi': hi,
        'h2_pass': lo > 0,
    })

h2_df = pd.DataFrame(h2_records)
h2_df.to_csv(DATA_DIR / 'bootstrap_h2.csv', index=False)
print('H2 results (B3 − M1 ΔRMSE):')
print(h2_df.to_string(index=False))
n_pass = h2_df['h2_pass'].sum()
print(f'\nH2 OUTCOME: {"SUPPORTED" if n_pass >= 2 else "NOT SUPPORTED"} ({n_pass}/4 metals pass)')

## 3. H3: M2 (genus-weighted) vs M3 (CWM) comparison

In [ ]:
m2_rmse = cv_all[cv_all['model'] == 'M2'].groupby('target')['rmse'].mean()
m3_rmse = cv_all[cv_all['model'] == 'M3'].groupby('target')['rmse'].mean()

h3_df = pd.DataFrame({
    'M2_rmse': m2_rmse,
    'M3_rmse': m3_rmse,
    'M2_beats_M3': m2_rmse < m3_rmse,
}).round(4)
print('H3: Genus-weighted (M2) vs CWM (M3):')
print(h3_df)
n_h3 = h3_df['M2_beats_M3'].sum()
print(f'\nH3 OUTCOME: {"SUPPORTED" if n_h3 >= 3 else "NOT SUPPORTED"} ({n_h3}/4 metals M2 < M3)')

## 4. Summary plot

In [ ]:
plot_rmse_comparison(
    cv_all,
    models=['B0', 'B1', 'B2', 'B3', 'B4', 'M1', 'M2', 'M3'],
    targets=[t for t in TARGETS if t in feature_matrix.columns],
    out_path=FIG_DIR / 'model_rmse_comparison.png',
)